In [5]:
import pandas as pd
import numpy as np

# generate by ChatGPT, with some manual edits
with open("data_asg1.txt", encoding="utf-8") as f:
    texts = f.readlines()

with open("labels_asg1.txt") as f:
    labels = f.read().split()

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df.head()

,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1


In [6]:
import re

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)       
    text = re.sub(r"@\w+", " ", text)          
    text = re.sub(r"#", " ", text)             
    text = re.sub(r"[^a-z0-9\s]", " ", text)    
    text = re.sub(r"\s+", " ", text).strip()    
    return text

df["clean_text"] = df["text"].apply(clean_tweet)


In [7]:
# 3)Sentence embeddings
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
X_sbert = sbert_model.encode(df["text"].tolist(), show_progress_bar=True)
y = df["label"].values
print("X_sbert shape:", X_sbert.shape) 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 299.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 144/144 [00:14<00:00, 10.13it/s]

X_sbert shape: (4601, 384)


In [8]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_scores_model_dense(X, y, model):
    accs, f1s, macro_f1s = [], [], []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf = model()
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred, pos_label="1"))
        macro_f1s.append(f1_score(y_test, y_pred, average="macro"))

    acc_mean = round(np.mean(accs), 4)
    f1_mean = round(np.mean(f1s), 4)
    macro_f1_mean = round(np.mean(macro_f1s), 4)
    return acc_mean, f1_mean, macro_f1_mean

print("SBERT + RandomForest:", cv_scores_model_dense(X_sbert, y, RandomForestClassifier))
print("SBERT + LinearSVC:", cv_scores_model_dense(X_sbert, y, LinearSVC))
print("SBERT + DecisionTree:", cv_scores_model_dense(X_sbert, y, DecisionTreeClassifier))


SBERT + RandomForest: (np.float64(0.6383), np.float64(0.5865), np.float64(0.6325))
SBERT + LinearSVC: (np.float64(0.6633), np.float64(0.6381), np.float64(0.6617))
SBERT + DecisionTree: (np.float64(0.5488), np.float64(0.5339), np.float64(0.548))
